# Clase 2 — Prompt Engineering como sistema

**Curso:** Arquitectura de Aplicaciones con IA Generativa · EscuelaIT  
**Duración:** 90 min · **Proveedor:** DeepSeek (vía API OpenAI-compatible)

> Objetivo: tratar el prompting como **sistema** — versionado, testeado, iterado con datos — no como arte oculto en un string.

Dominio conductor del notebook: un asistente de monitoreo electoral (inspirado en el proyecto `peru-elecciones`). Al final mapeamos estos patrones al código Go real.

## Agenda

Basada en el slide 35 de cierre de la clase 1:

0. **Setup** — cliente DeepSeek, `.env`, helper
1. **Anatomía rápida** — roles y `temperature` (repaso de clase 1)
2. **Técnicas** — zero-shot · few-shot · chain-of-thought · self-consistency
3. **System prompts como arquitectura** — rol · contexto · restricciones · formato
4. **Output estructurado** — JSON mode · Pydantic + Instructor · validación · truncamiento
5. **Pipeline multi-etapa** — extracción → clasificación → informe
6. **Puente al código Go** — los mismos patrones en `peru-elecciones`

---
## 0 · Setup

### 0.1 Instalación de dependencias

Ejecuta **una sola vez** (o desde terminal con `pip install -r requirements.txt`).

In [ ]:
# %pip install -q openai python-dotenv pydantic instructor rich

### 0.2 Variables de entorno

Copia `.env.example` a `.env` y rellena `DEEPSEEK_API_KEY`. El cliente `openai` habla con DeepSeek **cambiando solo `base_url`** — exactamente el patrón que vimos en la diapo 31 de clase 1.

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com",
)

MODEL = "deepseek-chat"        # conversación general
MODEL_REASONER = "deepseek-reasoner"  # CoT nativo — lo usaremos en §2.3

def ask(messages, *, model=MODEL, temperature=0.3, max_tokens=1024, **kwargs):
    """Wrapper mínimo — oculta la repetición, nada más."""
    return client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
        **kwargs,
    )

def say(resp):
    """Imprime respuesta + métricas."""
    msg = resp.choices[0].message.content
    u = resp.usage
    print(msg)
    print(f"\n— tokens: prompt={u.prompt_tokens} completion={u.completion_tokens} total={u.total_tokens}")
    print(f"— finish_reason: {resp.choices[0].finish_reason}")

### 0.3 Test de conexión

In [2]:
resp = ask([
    {"role": "system", "content": "Eres un asistente conciso experto en elecciones peruanas."},
    {"role": "user",   "content": "En una frase: ¿qué es la ONPE?"},
])
say(resp)

La ONPE es el organismo autónomo encargado de organizar y ejecutar los procesos electorales en el Perú.

— tokens: prompt=31 completion=27 total=58
— finish_reason: stop


---
## 1 · Anatomía rápida — roles y `temperature`

Repaso de clase 1:

- **`system`** — contrato del asistente (rol, reglas, formato). Lo veremos a fondo en §3.
- **`user`** — turno del humano.
- **`assistant`** — turno del modelo. Se incluye en `messages` cuando mantienes historial.
- **`tool`** — resultado de una tool invocada (lo veremos en clase 4).

`temperature` controla la aleatoriedad: **0 = casi determinista**, **0.7 = conversación**, **>1 = creativo/errático**.

In [4]:
mensajes = [
    {"role": "system", "content": "Eres un desarrollador senior. Máximo 12 palabras."},
    {"role": "user",   "content": "Dinos cuál es el mejor lenguaje de programación"},
]

for t in [0.0, 0.7, 1.5]:
    out = ask(mensajes, temperature=t).choices[0].message.content.strip()
    print(f"[t={t}]  {out}")

[t=0.0]  Python: versátil, legible, amplia comunidad.
[t=0.7]  Python: versátil, legible y ampliamente adoptado.
[t=1.5]  Python, por su versatilidad y comunidad.


**Observación para la clase:** corre la celda 2–3 veces. Con `t=0` el output apenas cambia; con `t=1.5` aparecen palabras raras o frases rotas. Regla simple: **clasificación/extracción → 0. Escritura creativa → 0.7–1.0. Brainstorm → 1.0–1.3.**

---
## 2 · Técnicas de prompting

Cuatro técnicas que cubren el 90% de los casos prácticos. Las vemos con el mismo dominio: clasificar tweets sobre el proceso electoral en 4 categorías — `CONFIANZA`, `DESCONFIANZA`, `APATIA`, `RECLAMO`.

### 2.1 Zero-shot

Describes la tarea **sin ejemplos**. Funciona cuando el modelo ya conoce la tarea (traducción, resumen, clasificación común).

In [6]:
tweet = "El conteo va lento, es un desastre"

out = ask([
    {"role": "system", "content": (
        "Clasifica tweets sobre el proceso electoral peruano en una de cuatro etiquetas: "
        "CONFIANZA, DESCONFIANZA, APATIA, RECLAMO. Responde SOLO la etiqueta."
    )},
    {"role": "user", "content": f"Tweet: {tweet}\nEtiqueta:"},
], temperature=0).choices[0].message.content.strip()

print(out)

DESCONFIANZA


### 2.2 Few-shot

Añades **3–5 ejemplos** entrada→salida. Eleva la consistencia de formato y ayuda en tareas donde las etiquetas tienen matices propios.

In [7]:
FEW_SHOT = """\
Tweet: "Hoy fui a votar temprano, todo rápido y ordenado"
Etiqueta: CONFIANZA

Tweet: "¿Por qué aún no hay resultados? Esto es una vergüenza."
Etiqueta: RECLAMO

Tweet: "No votaré, total son todos iguales"
Etiqueta: APATIA

Tweet: "El JNE cambió las reglas en medio del proceso, muy raro todo"
Etiqueta: DESCONFIANZA
"""

def clasificar(tweet: str) -> str:
    return ask([
        {"role": "system", "content": "Clasifica tweets. Devuelve SOLO la etiqueta en MAYÚSCULAS."},
        {"role": "user",   "content": f"{FEW_SHOT}\nTweet: \"{tweet}\"\nEtiqueta:"},
    ], temperature=0).choices[0].message.content.strip()

pruebas = [
    "El conteo va lento, mejor espero hasta mañana",
    "Muy ordenada la mesa 123 en Surco, felicitaciones",
    "Las actas no cuadran en mi local, algo raro pasó ahí",
    "Bah, da igual quién gane",
]
for t in pruebas:
    print(f"{clasificar(t):15s}  ← {t}")


DESCONFIANZA     ← El conteo va lento, mejor espero hasta mañana
CONFIANZA        ← Muy ordenada la mesa 123 en Surco, felicitaciones
DESCONFIANZA     ← Las actas no cuadran en mi local, algo raro pasó ahí
APATIA           ← Bah, da igual quién gane


**Regla de dedo:** si cambias de few-shot a zero-shot y la métrica sube, **los ejemplos estaban mal elegidos** (sesgaban al modelo). Few-shot no es gratis: consume tokens y puede anclar al modelo a patrones superficiales de tus ejemplos.

### 2.3 Chain-of-Thought (CoT)

La instrucción mágica: **"razona paso a paso"**. Mejora tareas de aritmética, lógica y decisiones multi-criterio. Coste: más tokens de salida.

**Modelos de razonamiento** (`deepseek-reasoner`, `o-series` de OpenAI, Gemini Thinking) hacen CoT **nativo** — no necesitas pedirlo. Veremos ambas variantes.

In [8]:
pregunta_aritm = (
    "En segunda vuelta hay 18.2M de votos válidos. El candidato A tiene 50.8% y el B 49.2%. "
    "Un 2.1% de actas aún no se computan. ¿Cuántos votos de diferencia hay y puede revertirse?"
)

# Sin CoT — respuesta directa, suele equivocarse
sin_cot = ask(
    [{"role": "user", "content": pregunta_aritm + " Responde en una frase, SIN explicar."}],
    temperature=0,
).choices[0].message.content.strip()
print("▼ SIN CoT\n" + sin_cot)

▼ SIN CoT
La diferencia es de aproximadamente 291 200 votos y, con el 2.1% de actas pendientes, la reversión es estadísticamente improbable.


In [9]:
# Con CoT explícito
con_cot = ask(
    [{"role": "user", "content": pregunta_aritm + " Razona paso a paso y da al final la respuesta entre <resp></resp>."}],
    temperature=0,
    max_tokens=2048,
).choices[0].message.content.strip()
print("▼ CON CoT explícito\n" + con_cot)


▼ CON CoT explícito
Vamos paso a paso.  

---

**1. Datos conocidos**  
- Votos válidos contabilizados hasta ahora: \( 18.2 \) millones.  
- Porcentaje de A: \( 50.8\% \)  
- Porcentaje de B: \( 49.2\% \)  
- Falta por contar: \( 2.1\% \) de las actas totales (esto no es el 2.1% de los votos ya contados, sino del total de actas; pero normalmente se interpreta como que falta por contar aproximadamente el 2.1% del total de votos válidos finales).  

---

**2. Diferencia actual de votos**  
Votos de A:  
\[
0.508 \times 18.2 \text{ millones} = 9.2456 \text{ millones}
\]  
Votos de B:  
\[
0.492 \times 18.2 \text{ millones} = 8.9544 \text{ millones}
\]  
Diferencia actual:  
\[
9.2456 - 8.9544 = 0.2912 \text{ millones} = 291{,}200 \text{ votos}
\]  

---

**3. Interpretación del 2.1% de actas faltantes**  
Si el 2.1% de actas faltantes corresponde al 2.1% del total de votos válidos finales, primero debemos estimar el total final de votos válidos.  

Sea \( T \) = total de votos válidos fin

In [10]:
# Con modelo de razonamiento — el CoT está oculto en `reasoning_content`
resp_reasoner = ask(
    [{"role": "user", "content": pregunta_aritm}],
    model=MODEL_REASONER,
    temperature=0,
    max_tokens=2048,
)
msg = resp_reasoner.choices[0].message
reasoning = getattr(msg, "reasoning_content", None)
print("▼ RAZONAMIENTO (oculto para el usuario final)\n")
print((reasoning or "")[:600], "...\n")
print("▼ RESPUESTA FINAL\n")
print(msg.content)

▼ RAZONAMIENTO (oculto para el usuario final)

Vamos paso a paso.

**1. Entender los datos:**
- Votos válidos en segunda vuelta: 18.2 millones.
- Porcentajes reportados (sobre votos válidos ya contados? O sobre el total estimado?): dice "El candidato A tiene 50.8% y el B 49.2%". Normalmente, cuando se dan porcentajes así en un conteo parcial, se refieren al porcentaje de los votos válidos contados hasta ese momento. Pero el enunciado dice: "En segunda vuelta hay 18.2M de votos válidos. El candidato A tiene 50.8% y el B 49.2%. Un 2.1% de actas aún no se computan." Esto puede interpretarse de dos maneras: 
    a) Los 18.2M son el total de vo ...

▼ RESPUESTA FINAL

En el escrutinio actual, con el 97.9% de las actas contadas (correspond


### 2.4 Self-consistency

Generas **N muestras** con `temperature>0` y te quedas con la **más votada**. Pagas N× el coste a cambio de robustez en casos ambiguos. Útil cuando el espacio de salida es pequeño (clasificación, extracción de 1 campo).

In [11]:
from collections import Counter

def votar_etiqueta(tweet: str, n: int = 5, temperature: float = 1.1) -> tuple[str, list[str]]:
    muestras = []
    for _ in range(n):
        r = ask([
            {"role": "system", "content": "Clasifica en: CONFIANZA, DESCONFIANZA, APATIA, RECLAMO. Solo etiqueta."},
            {"role": "user",   "content": f"{FEW_SHOT}\nTweet: \"{tweet}\"\nEtiqueta:"},
        ], temperature=temperature).choices[0].message.content.strip()
        muestras.append(r)
    ganadora, _ = Counter(muestras).most_common(1)[0]
    return ganadora, muestras

# Tweet con señales contradictorias (elogio aparente + queja por retraso) para forzar ambigüedad
ambiguo = "Qué maravilla este ordenado conteo, solo 3 horas de retraso, felicitaciones ONPE"
ganadora, todas = votar_etiqueta(ambiguo, n=5)
print(f"Tweet: {ambiguo!r}")
print("Muestras :", todas)
print("Ganadora :", ganadora)


Tweet: 'Qué maravilla este ordenado conteo, solo 3 horas de retraso, felicitaciones ONPE'
Muestras : ['RECLAMO', 'RECLAMO', 'RECLAMO', 'RECLAMO', 'RECLAMO']
Ganadora : RECLAMO


---
## 3 · System prompts como arquitectura

El `system` no es una oración suelta — es un **contrato**. Estructura recomendada en cuatro bloques:

| Bloque | Qué contiene | Qué problema ataca |
|---|---|---|
| **ROL** | Quién es el asistente, su dominio | Desenfoque temático |
| **CONTEXTO** | Fecha, datos disponibles, fuentes | Alucinación por falta de anclaje |
| **RESTRICCIONES** | Qué NO hacer, límites duros | Output no regulatorio / tóxico / inventado |
| **FORMATO** | Estructura de salida esperada | Parseo aguas abajo |

Comparemos un prompt pobre y uno estructurado con la misma pregunta.

In [12]:
SYSTEM_POBRE = "Eres un asistente sobre elecciones peruanas."

out = ask([
    {"role": "system", "content": SYSTEM_POBRE},
    {"role": "user",   "content": "¿Quién va ganando en Arequipa?"},
], temperature=0).choices[0].message.content

print("▼ SYSTEM POBRE\n")
print(out)

▼ SYSTEM POBRE

Para conocer los resultados electorales oficiales y actualizados en Arequipa o cualquier otra región del Perú, debes consultar directamente las fuentes oficiales, ya que los datos pueden cambiar rápidamente durante los procesos de conteo.

Las entidades oficiales para verificar resultados son:

1. **Oficina Nacional de Procesos Electorales (ONPE)** - Publica resultados preliminares en tiempo real durante las elecciones.
   - Sitio web: [www.onpe.gob.pe](https://www.onpe.gob.pe)
   - Aplicación móvil: "ONPE Resultados"

2. **Jurado Nacional de Elecciones (JNE)** - Proclama los resultados oficiales finales.
   - Sitio web: [www.jne.gob.pe](https://www.jne.gob.pe)

**Recomendación:**  
Accede a estos portales oficiales y busca la sección de "Resultados Electorales" o "Elecciones" para obtener información precisa, desagregada por región, provincia y distrito. Los medios de comunicación serios también suelen reflejar estos datos oficiales en sus coberturas.

*Nota: Como asis

In [13]:
SYSTEM_PRO = """\
# REGLA CRÍTICA — LEER PRIMERO
Bajo NINGUNA circunstancia inventes candidatos, partidos, regiones, fechas ni cifras.
Si en el mensaje del usuario no recibes datos concretos (porcentajes, nombres citados, actas),
tu ÚNICA respuesta permitida es literalmente:
"No dispongo de ese dato en mi contexto actual."
No añadas nada más. No intentes ser útil rellenando con conocimiento general ni histórico.

# ROL
Eres ONPE-Asistente, un asistente oficial sobre el conteo electoral 2026 en Perú.
Te consultan ciudadanos, prensa y equipos de campaña.

# CONTEXTO
- Solo puedes usar los datos que el usuario te pase explícitamente en su mensaje.
- Fecha actual: 21 de abril de 2026. Escrutinio en curso.

# RESTRICCIONES
- Nunca opines políticamente ni recomiendes candidatos.
- Nunca proyectes un ganador si el porcentaje escrutado es inferior al 95%.
- Nunca menciones fuentes no oficiales.
- Máximo 3 oraciones por respuesta.

# FORMATO (cuando SÍ tienes datos)
- Oración 1: respuesta directa.
- Oración 2 (si aplica): cifra con `escrutado X%` entre paréntesis.
- Oración 3 (si aplica): advertencia de provisionalidad.
"""

# Turno 1 — pregunta SIN contexto
out_sin = ask([
    {"role": "system", "content": SYSTEM_PRO},
    {"role": "user",   "content": "¿Quién va ganando en Arequipa?"},
], temperature=0).choices[0].message.content

print("▼ SYSTEM ESTRUCTURADO — pregunta SIN contexto\n")
print(out_sin)

# Turno 2 — misma pregunta pero con datos concretos inyectados
out_con = ask([
    {"role": "system", "content": SYSTEM_PRO},
    {"role": "user",   "content": (
        "Datos oficiales ONPE · Arequipa · actualizado 21/04/2026 10:00\n"
        "- Ayala (Fuerza Regional): 38.4%\n"
        "- Rivas (Arequipa Unida): 36.1%\n"
        "- Escrutado: 72%\n\n"
        "¿Quién va ganando en Arequipa?"
    )},
], temperature=0).choices[0].message.content

print("\n▼ SYSTEM ESTRUCTURADO — misma pregunta, ahora CON contexto\n")
print(out_con)


▼ SYSTEM ESTRUCTURADO — pregunta SIN contexto

No dispongo de ese dato en mi contexto actual.

▼ SYSTEM ESTRUCTURADO — misma pregunta, ahora CON contexto

En Arequipa, con el 72% de actas escrutadas, va ganando el candidato Ayala de Fuerza Regional.
Los resultados son provisionales y pueden cambiar.


**Lo que acabamos de ver:**

- **Sin contexto** el system estructurado responde literal *"no dispongo de ese dato"* en vez de inventar candidatos. El system pobre, en cambio, fabrica hechos con mucha seguridad.
- **Con contexto** el mismo system cambia de comportamiento y responde en el formato pedido, citando las cifras que le pasaste.

El mismo prompt ajusta su comportamiento según lo que el *user* le entrega. Esto es justo lo que quieres para un asistente que atiende prensa y ciudadanos: **no opina si no le diste datos**.

> **Patrón:** el system prompt vive en un archivo versionado (`.md`/`.txt`), se testea con un set de preguntas adversariales (¿y si le pregunto algo sin contexto? ¿y si le pido proyectar un ganador con 40% escrutado?), y se cambia con el mismo rigor que cualquier otro código de producción.


---
## 4 · Output estructurado

Cuando el LLM es un componente de software (no un chat), su salida tiene que ser **parseable por código**. Tres niveles de rigor:

### 4.1 JSON mode nativo

DeepSeek soporta `response_format={"type": "json_object"}`: **garantiza JSON válido sintácticamente**, no valida esquema. Útil, pero insuficiente por sí solo.

In [16]:
resp = ask([
    {"role": "system", "content": (
        "Extrae entidades del tweet. Devuelve JSON con claves: "
        "candidato (string|null), region (string|null), mesa (string|null), porcentaje (number|null)."
    )},
    {"role": "user", "content": "En la mesa 123 de Surco, el candidato López va con 42.1%."},
], response_format={"type": "json_object"}, temperature=0)

data = json.loads(resp.choices[0].message.content)
print(data)

{'candidato': 'López', 'region': 'Surco', 'mesa': '123', 'porcentaje': 42.1}


**Gotcha:** el modelo puede devolver claves extra, o tipos incorrectos (porcentaje como string `"42.1%"`). El JSON es válido; el dato no.

### 4.2 Pydantic + Instructor

`instructor` envuelve el cliente OpenAI y:

1. Convierte tu modelo Pydantic en un schema para el prompt.
2. Valida la respuesta contra el schema.
3. Si falla, **reintenta mandando el error al modelo** para que se corrija.

Resultado: tu función `llm_call(...) -> MiPydanticModel` devuelve tipos Python — no strings.

In [17]:
from pydantic import BaseModel, Field
from typing import Literal, Optional
import instructor

class ResultadoMesa(BaseModel):
    candidato: str = Field(description="Apellido o nombre completo del candidato")
    region: str
    mesa: Optional[str] = Field(default=None, description="Número de mesa; null si no se menciona")
    porcentaje: float = Field(ge=0, le=100)
    tendencia: Literal["subiendo", "bajando", "estable", "desconocida"]

icli = instructor.from_openai(
    OpenAI(api_key=os.environ["DEEPSEEK_API_KEY"], base_url="https://api.deepseek.com"),
    mode=instructor.Mode.MD_JSON,  # modo compatible con DeepSeek
)

resultado = icli.chat.completions.create(
    model=MODEL,
    response_model=ResultadoMesa,
    messages=[
        {"role": "system", "content": "Extrae datos electorales estructurados."},
        {"role": "user",   "content": (
            "En la mesa 00123 de Arequipa, López va con 42.1% y ha ido ganando terreno desde las 18h."
        )},
    ],
    max_retries=2,
)
print(type(resultado))
print(resultado.model_dump_json(indent=2))

<class '__main__.ResultadoMesa'>
{
  "candidato": "López",
  "region": "Arequipa",
  "mesa": "00123",
  "porcentaje": 42.1,
  "tendencia": "subiendo"
}


### 4.3 Validación con retry manual

Si no quieres depender de `instructor`, el patrón equivalente en tres líneas:

In [18]:
from pydantic import ValidationError

def pedir_json_tipado(messages, schema_cls, intentos: int = 3):
    historial = list(messages)
    last_error = None
    for i in range(intentos):
        if last_error:
            historial.append({"role": "user", "content": (
                f"Tu respuesta anterior no validó. Error: {last_error}. "
                f"Devuelve SOLO un JSON que respete el schema."
            )})
        resp = ask(historial, response_format={"type": "json_object"}, temperature=0)
        raw = resp.choices[0].message.content
        historial.append({"role": "assistant", "content": raw})
        try:
            return schema_cls.model_validate_json(raw)
        except ValidationError as e:
            last_error = str(e)[:300]
    raise ValueError(f"No validó tras {intentos} intentos: {last_error}")

# Schema con una exigencia difícil
schema_prompt = ResultadoMesa.model_json_schema()
r = pedir_json_tipado([
    {"role": "system", "content": f"Extrae y responde JSON con este schema: {json.dumps(schema_prompt)}"},
    {"role": "user",   "content": "En Puno, la candidata Quispe subió a 55.4% — mesa 88-A."},
], ResultadoMesa)

print(r.model_dump_json(indent=2))

{
  "candidato": "Quispe",
  "region": "Puno",
  "mesa": "88-A",
  "porcentaje": 55.4,
  "tendencia": "subiendo"
}


### 4.4 Truncamiento

Si `finish_reason == "length"`, el modelo **se quedó a medias** porque topó con `max_tokens`. Esto rompe JSON estructurado (queda sin cerrar). Hay que detectarlo — no confiar en que el parser fallará con un mensaje claro.

In [19]:
resp = ask([
    {"role": "user", "content": "Escribe una crónica detallada de 5000 palabras sobre la jornada electoral."}
], max_tokens=60)

print("finish_reason :", resp.choices[0].finish_reason)
print("contenido     :", resp.choices[0].message.content[:200], "...")

if resp.choices[0].finish_reason == "length":
    print("\n⚠ Salida truncada — en producción, o subes max_tokens, o llamas de nuevo pidiendo continuar.")

finish_reason : length
contenido     : # Crónica de una jornada electoral: El día en que la democracia respiró

**Amanecer en la urna**

Las primeras luces del alba tiñeron de azul pálido los edificios de la ciudad cuando Elena, presidenta ...

⚠ Salida truncada — en producción, o subes max_tokens, o llamas de nuevo pidiendo continuar.


---
## 5 · Pipeline multi-etapa

Ahora juntamos todo: un **mini sistema de monitoreo electoral**. Entrada: un tweet. Salida: un informe tipado con acción recomendada.

Tres etapas — cada una es una función pura `str → BaseModel`, cada una testeable por separado:

```
tweet crudo
   │
   ▼
[Etapa 1] extracción de entidades ─→ EntidadesTweet
   │
   ▼
[Etapa 2] clasificación + justificación ─→ Clasificacion
   │
   ▼
[Etapa 3] informe con acción recomendada ─→ Informe
```

**Por qué dividir en etapas** en vez de un solo prompt gigante:

- Cada etapa tiene un system prompt pequeño y enfocado.
- Cachear entre etapas es trivial (hash del input).
- Puedes cambiar de modelo por etapa (barato para extracción, caro para razonamiento).
- Cada schema es un test. Si la etapa 1 falla, no se propaga basura.

In [ ]:
from pydantic import BaseModel, Field, conint
from typing import Literal

class EntidadesTweet(BaseModel):
    candidatos: list[str] = Field(default_factory=list)
    regiones: list[str] = Field(default_factory=list)
    numeros_mencionados: list[str] = Field(default_factory=list, description="Porcentajes, nº mesa, nº actas, etc.")
    menciona_fraude: bool
    menciona_violencia: bool

class Clasificacion(BaseModel):
    etiqueta: Literal["CONFIANZA", "DESCONFIANZA", "APATIA", "RECLAMO"]
    confianza: float = Field(ge=0, le=1, description="Qué tan seguro estás de la etiqueta")
    razones: list[str] = Field(min_length=1, max_length=3)

class Informe(BaseModel):
    resumen: str = Field(max_length=240)
    accion_recomendada: Literal["ignorar", "monitorear", "escalar_a_humano", "derivar_a_prensa"]
    nivel_urgencia: conint(ge=1, le=5)

In [ ]:
def etapa_extraccion(tweet: str) -> EntidadesTweet:
    return icli.chat.completions.create(
        model=MODEL,
        response_model=EntidadesTweet,
        messages=[
            {"role": "system", "content": (
                "Eres un extractor de entidades electorales. Lee el tweet y devuelve el schema. "
                "No interpretes ni opines — solo lo que aparece explícito."
            )},
            {"role": "user", "content": tweet},
        ],
        max_retries=2,
    )

def etapa_clasificacion(tweet: str, entidades: EntidadesTweet) -> Clasificacion:
    contexto = f"Entidades detectadas: {entidades.model_dump_json()}"
    return icli.chat.completions.create(
        model=MODEL,
        response_model=Clasificacion,
        messages=[
            {"role": "system", "content": (
                "Clasifica el tweet en CONFIANZA, DESCONFIANZA, APATIA o RECLAMO. "
                "Justifica con 1–3 razones cortas basadas en las entidades y el texto."
            )},
            {"role": "user", "content": f"{contexto}\n\nTweet: {tweet}"},
        ],
        max_retries=2,
    )

def etapa_informe(tweet: str, entidades: EntidadesTweet, clasif: Clasificacion) -> Informe:
    contexto = json.dumps({
        "entidades": entidades.model_dump(),
        "clasificacion": clasif.model_dump(),
    }, ensure_ascii=False, indent=2)
    return icli.chat.completions.create(
        model=MODEL,
        response_model=Informe,
        messages=[
            {"role": "system", "content": (
                "Eres un analista de monitoreo electoral. Produce un informe breve y decide una acción. "
                "Reglas de escalamiento: "
                " - menciona_fraude=true o menciona_violencia=true → nivel_urgencia ≥ 4 y acción 'escalar_a_humano'. "
                " - etiqueta=APATIA → 'ignorar' (urgencia 1–2)."
            )},
            {"role": "user", "content": f"Input:\n{contexto}\n\nTweet original:\n{tweet}"},
        ],
        max_retries=2,
    )

In [ ]:
def pipeline(tweet: str) -> dict:
    ent = etapa_extraccion(tweet)
    cls = etapa_clasificacion(tweet, ent)
    inf = etapa_informe(tweet, ent, cls)
    return {"entidades": ent, "clasificacion": cls, "informe": inf}

ejemplos = [
    "ALERTA en Puno mesa 00421: llegaron las ánforas abiertas. Candidato Vargas pierde votos!!!",
    "Muy ordenada la votación en mi colegio de Miraflores, felicitaciones a la ONPE.",
    "Da igual quién gane, todos son lo mismo",
]

for tw in ejemplos:
    print("=" * 70)
    print("TWEET:", tw)
    r = pipeline(tw)
    print("\n[entidades]    ", r["entidades"].model_dump_json())
    print("[clasificación]", r["clasificacion"].model_dump_json())
    print("[informe]      ", r["informe"].model_dump_json())

### Observaciones del pipeline

- **Coste y latencia** escalan lineal con el número de etapas. Tres etapas ≈ 3× tokens y 3× latencia vs un solo prompt. Se compensa con cache y con poder usar modelos más baratos en etapas simples.
- **Cada etapa es un test**. `etapa_extraccion` con un fixture de 20 tweets te dice si un cambio de prompt mejoró o rompió algo — sin correr el pipeline completo.
- **Los schemas son la interfaz**. Si mañana cambias `etapa_clasificacion` por un modelo fine-tuned local, la etapa 3 no se entera.
- **Cuando el pipeline deja de ser lineal** — con ramas, loops, memoria, herramientas — dejamos de llamarlo "pipeline" y empezamos a llamarlo **orquestación**. Eso es la clase 3.

---
## 6 · Puente al código Go de `peru-elecciones`

Ahora saltamos del notebook al editor. Abrimos `/Users/manduinca/Projects/deepskill/peru-elecciones` y buscamos **los mismos patrones que acabamos de ver**:

| Concepto del notebook | Dónde aparece en Go |
|---|---|
| Cliente OpenAI-compatible con `base_url` configurable | Paquete cliente LLM (abstracción agnóstica) |
| System prompt versionado con rol/contexto/restricciones | Template del prompt del asistente electoral |
| Inyección de contexto en tiempo real | Queries a SQLite que se interpolan antes de la llamada |
| `messages` como estructura de primera clase | Struct + builder de mensajes |
| `temperature` / `max_tokens` como config, no hardcoded | Config de servicio |
| Manejo de `finish_reason` y errores | Wrapper del cliente |

*(Durante la clase, pasamos a VS Code; esta celda queda como índice para que los alumnos revisen después.)*

---
## Cierre — tres ideas para llevar

1. **Los prompts son código.** Se versionan, testean, iteran con datos. Un system prompt que no tiene sus 20 preguntas adversariales de regresión no está listo para producción.
2. **Estructura > `temperature=0`.** Un system prompt con rol/contexto/restricciones/formato apaga más alucinaciones que bajar `temperature`.
3. **Structured output es la frontera entre demo y producto.** JSON mode abre la puerta; Pydantic + Instructor (o retry manual con validación) la cierra.

### Preview clase 3 — Patrones de arquitectura con LLMs

El pipeline lineal es el punto de partida. En la clase 3 lo rompemos:

- **RAG** — cuando el contexto no cabe en el prompt, lo recuperas dinámicamente.
- **Memoria** — cuando el pipeline necesita acordarse de turnos anteriores.
- **Orquestación** — cuando las etapas tienen ramas, loops y decisiones del propio modelo.
- **Tool use / MCP** — cuando el modelo no genera texto sino acciones.